> **Статус: ПРЕДЛОЖЕННОЕ РАСШИРЕНИЕ (research branch), не входит в основной бенчмарк.**
>
> Лидерборды и все заявленные результаты (разделы 3 и 5) используют обученную модель
> **EVT-NeuralSSM БЕЗ топологической регуляризации**. Данный ноутбук описывает и проверяет
> *идею* добавления `λ·L_topology` и по умолчанию (`RUN_TRAINING=False`) выдаёт только
> метод + диагностику/прокси на готовой модели — он **не переобучает** модель и **не меняет**
> `models/`. Чтобы получить фактически обученную topology-regularized модель, поставьте
> `RUN_TRAINING=True`; полученный артефакт следует отчитывать **отдельно** как расширение,
> а не смешивать с основным лидербордом.

# 4.3 EVT-NeuralSSM + топологическая регуляризация

Идея этого ноутбука: не менять старые архитектуры и не трогать DPI/DPI-EVT, а аккуратно добавить исследовательскую ветку только для EVT-NeuralSSM:

$$
L = L_{data} + L_{physics} + \lambda L_{topology}.
$$

`L_topology` сохраняет структуру траекторий: близкие по наблюдаемой динамике PPR/CSR опыты должны оставаться близкими в пространстве предсказанных состояний EVT (`PPR`, `Damage`, `Event trigger`, memory-state). Это даёт комбинацию:

**Physics + State Space + Event Trigger + Topology**.

Практически это differentiable proxy для топологии Vietoris-Rips 0-скелета / neighborhood graph: в мини-батче сравниваются матрицы попарных расстояний между наблюдаемыми траекториями и между предсказанными state-space траекториями. Persistent homology используется ниже только как опциональная пост-фактум диагностика, если установлены `ripser/kmapper/umap`; регуляризатор от них не зависит.


In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import sys
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

REPO = Path.cwd()
if not (REPO / 'src').exists():
    REPO = Path.cwd().parents[1]
sys.path.insert(0, str(REPO / 'src'))

from liquefaction_ai import load_population_artifact, prepare_benchmark_dataset
from liquefaction_ai.config import set_global_seed
from liquefaction_ai.data.splits import iterate_minibatches
from liquefaction_ai.evaluation.metrics import collect_outputs, compute_metrics
from liquefaction_ai.models.evt_ssm import EVTNeuralSSM
from liquefaction_ai.training import read_hyperparams, train_model, save_trained_model
from liquefaction_ai.training.persistence import load_weights_into
from liquefaction_ai.viz.figure_io import save_figure

set_global_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


## Конфигурация эксперимента

По умолчанию ноутбук **не дообучает** модель, чтобы ячейки можно было безопасно прогонять как проверку идеи. Для реального эксперимента поменяйте `RUN_TRAINING = True`: будет создана отдельная модель `models/evt_ssm_toporeg`, не затрагивая `models/evt_ssm`, DPI-Flow и DPI-EVT.


In [ ]:
ARTIFACT_DIR = REPO / 'data' / 'dataset'
BASE_MODEL_NAME = 'evt_ssm'
TOPO_MODEL_NAME = 'evt_ssm_toporeg'

RUN_TRAINING = False          # research-флаг: True → отдельный topology-regularized артефакт
                              # (НЕ входит в основной лидерборд); False → только метод+диагностика
TOPOLOGY_LAMBDA = 0.03        # λ перед L_topology
TOPOLOGY_K = 12               # локальные соседи target-графа
TOPOLOGY_POINTS = 18          # якорные точки по времени для signatures
TOPOLOGY_EPOCHS = 2           # быстрый исследовательский fine-tune, не production retrain

population, cfg = load_population_artifact(ARTIFACT_DIR)
cfg.batch_size = min(cfg.batch_size, 128)
benchmark = prepare_benchmark_dataset(population, cfg, device)

hp = read_hyperparams(REPO / 'models', BASE_MODEL_NAME)
model_kwargs = dict(hp['model_kwargs'])
base_evt = EVTNeuralSSM(**model_kwargs).to(device)
base_evt = load_weights_into(base_evt, REPO / 'models', BASE_MODEL_NAME, device)

{split: benchmark[split]['static'].shape[0] for split in ['train', 'val', 'test']}, model_kwargs


## Дифференцируемый топологический proxy

Мы не оптимизируем persistent diagrams напрямую: это тяжело, шумно и не является гладкой функцией параметров модели. Вместо этого сохраняем локальную геометрию траекторий в батче.

1. Строим наблюдаемую signature-геометрию из `r_obs`, `cummax(r_obs)`, `CSR`.
2. Строим state-space signature модели из `PPR`, `Damage`, `trigger`, `c`.
3. Сравниваем нормированные попарные расстояния `D_data` и `D_state`.
4. Усиливаем локальную топологию весами kNN-графа: близкие траектории получают больший вес.

Это именно структурная регуляризация, а не ещё один pointwise-RMSE: она говорит модели сохранять форму облака траекторий.


In [ ]:
def _anchor_index(seq_len: int, n_points: int, device: torch.device) -> torch.Tensor:
    n = min(seq_len, n_points)
    return torch.linspace(0, seq_len - 1, n, device=device).round().long().unique()


def _fill_invalid_with_last(x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Заполнить невалидный хвост последним валидным значением, сохраняя форму кривой."""
    valid_count = mask.sum(dim=1).long().clamp(min=1)
    last_idx = (valid_count - 1).clamp(max=x.shape[1] - 1)
    last = x[torch.arange(x.shape[0], device=x.device), last_idx]
    return torch.where(mask > 0, x, last[:, None])


def _standardize_columns(x: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    return (x - x.mean(dim=0, keepdim=True)) / x.std(dim=0, keepdim=True).clamp_min(eps)


def observed_trajectory_signature(batch: Dict[str, torch.Tensor], n_points: int = 18) -> torch.Tensor:
    """Target signature: наблюдаемая геометрия PPR/CSR без латентных истин."""
    idx = _anchor_index(batch['r_obs'].shape[1], n_points, batch['r_obs'].device)
    r = _fill_invalid_with_last(batch['r_obs'], batch['mask'])[:, idx]
    csr = batch['csr'][:, idx]
    running = torch.cummax(r, dim=1).values
    slope = torch.diff(r, dim=1, prepend=r[:, :1])
    return torch.cat([r, running, csr, slope], dim=1)


def predicted_state_signature(outputs: Dict[str, torch.Tensor], n_points: int = 18) -> torch.Tensor:
    """State-space signature: Physics + State Space + Event Trigger."""
    r = outputs['traj_mean']
    idx = _anchor_index(r.shape[1], n_points, r.device)
    parts = [r[:, idx]]
    for key in ['z', 'g', 'c']:
        if key in outputs:
            parts.append(outputs[key][:, idx])
    parts.append(torch.diff(r, dim=1, prepend=r[:, :1])[:, idx])
    return torch.cat(parts, dim=1)


def pairwise_topology_loss(
    pred_signature: torch.Tensor,
    target_signature: torch.Tensor,
    k: int = 12,
    eps: float = 1e-6,
) -> torch.Tensor:
    """
    Differentiable neighborhood-geometry loss.

    Сравнивает матрицы попарных расстояний target/predicted signatures. Весовая матрица строится
    по kNN target-графу, поэтому сильнее сохраняются локальные компоненты и близкие режимы.
    """
    n = pred_signature.shape[0]
    if n < 3:
        return pred_signature.new_zeros(())

    pred = _standardize_columns(pred_signature)
    target = _standardize_columns(target_signature).detach()
    d_pred = torch.cdist(pred, pred)
    d_target = torch.cdist(target, target)

    target_scale = d_target[d_target > 0].median().clamp_min(eps)
    pred_scale = d_pred.detach()[d_pred.detach() > 0].median().clamp_min(eps)
    d_target = d_target / target_scale
    d_pred = d_pred / pred_scale

    with torch.no_grad():
        kk = min(max(int(k), 1), n - 1)
        nn_idx = torch.topk(d_target, k=kk + 1, largest=False).indices[:, 1:]
        local = torch.zeros_like(d_target)
        local.scatter_(1, nn_idx, 1.0)
        local = torch.maximum(local, local.T)
        sigma = d_target[d_target > 0].median().clamp_min(eps)
        weights = torch.exp(-d_target / sigma) * local
        weights = weights * (1.0 - torch.eye(n, device=weights.device, dtype=weights.dtype))

    return ((d_pred - d_target).pow(2) * weights).sum() / weights.sum().clamp_min(eps)


def topology_regularization(outputs: Dict[str, torch.Tensor], batch: Dict[str, torch.Tensor],
                            k: int = 12, n_points: int = 18) -> torch.Tensor:
    target = observed_trajectory_signature(batch, n_points=n_points)
    pred = predicted_state_signature(outputs, n_points=n_points)
    return pairwise_topology_loss(pred, target, k=k)


## Unit-тесты регуляризатора

Эти проверки быстрые и локальные: нулевая ошибка на одинаковых signatures, инвариантность к совместной перестановке батча и наличие градиента до predicted-state signature.


In [ ]:
def run_topology_unit_tests() -> None:
    g = torch.Generator(device='cpu').manual_seed(7)
    target = torch.randn(16, 24, generator=g)
    pred = target.clone().requires_grad_(True)

    same = pairwise_topology_loss(pred, target, k=5)
    assert same.item() < 1e-8, same.item()
    same.backward()
    assert torch.isfinite(pred.grad).all()

    perm = torch.randperm(target.shape[0], generator=g)
    permuted = pairwise_topology_loss(target[perm].clone(), target[perm], k=5)
    assert permuted.item() < 1e-8, permuted.item()

    distorted = pairwise_topology_loss(target.flip(0).clone(), target, k=5)
    assert distorted.item() > 1e-4, distorted.item()

    batch = next(iterate_minibatches(benchmark['train'], 24, device, shuffle=False))
    base_evt.eval()
    with torch.no_grad():
        out = base_evt.forward_batch(batch)
    loss = topology_regularization(out, batch, k=TOPOLOGY_K, n_points=TOPOLOGY_POINTS)
    assert torch.isfinite(loss), loss

run_topology_unit_tests()
print('OK: topology regularizer unit tests passed')


## EVTTopoSSM: локальная исследовательская обёртка

Никаких изменений в `src/liquefaction_ai/models/evt_ssm.py`: класс ниже живёт только в ноутбуке. Он наследует исходный EVT-NeuralSSM и добавляет к уже существующему loss новый член `λ L_topology`.


In [ ]:
class EVTTopoSSM(EVTNeuralSSM):
    def __init__(self, *args, topology_lambda: float = 0.03, topology_k: int = 12,
                 topology_points: int = 18, **kwargs):
        super().__init__(*args, **kwargs)
        self.topology_lambda = float(topology_lambda)
        self.topology_k = int(topology_k)
        self.topology_points = int(topology_points)

    def compute_loss(self, batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        outputs = super().compute_loss(batch)
        base_loss = outputs['loss']
        topo_loss = topology_regularization(outputs, batch, k=self.topology_k, n_points=self.topology_points)
        outputs['base_loss'] = base_loss.detach()
        outputs['topology_loss'] = topo_loss.detach()
        outputs['loss'] = base_loss + self.topology_lambda * topo_loss
        return outputs


def make_toporeg_model(load_evt_weights: bool = True) -> EVTTopoSSM:
    m = EVTTopoSSM(**model_kwargs, topology_lambda=TOPOLOGY_LAMBDA,
                   topology_k=TOPOLOGY_K, topology_points=TOPOLOGY_POINTS).to(device)
    if load_evt_weights:
        m = load_weights_into(m, REPO / 'models', BASE_MODEL_NAME, device)
        m.train()
    return m

# Smoke test: loss содержит topology_loss и остаётся дифференцируемым.
smoke = make_toporeg_model(load_evt_weights=True)
batch = next(iterate_minibatches(benchmark['train'], 16, device, shuffle=False))
loss_dict = smoke.compute_loss(batch)
assert 'topology_loss' in loss_dict and torch.isfinite(loss_dict['loss'])
print({k: float(loss_dict[k].detach().cpu()) for k in ['base_loss', 'topology_loss', 'loss']})


## Метрики сохранения структуры

Для оценки используем не только стандартные `N_liq`, `Brier`, `PPR RMSE`, но и топологические diagnostics:

- `topo_distance_corr`: корреляция попарных расстояний `D_data` и `D_state`.
- `topo_knn_overlap`: доля совпадающих kNN-соседей в observed trajectory graph и predicted state graph.
- `topo_trustworthiness`: trustworthiness embedding-like score, если доступен sklearn.
- `topo_h0_death_l2`: dependency-free H0 persistence proxy. Сравниваем sorted death times MST/union-find для observed/state distance matrices; меньше значит ближе структура компонент связности.

Эти метрики отвечают на вопрос: сохранила ли модель форму облака траекторий, а не просто попала pointwise в PPR.


In [ ]:
def _numpy_signature_from_split(split: Dict[str, object], outputs: Dict[str, np.ndarray], n_points: int = 18):
    idx = np.unique(np.rint(np.linspace(0, outputs['traj_mean'].shape[1] - 1, min(n_points, outputs['traj_mean'].shape[1]))).astype(int))
    r_obs = split['r_obs'].detach().cpu().numpy()
    mask = split['mask'].detach().cpu().numpy()
    csr = split['csr'].detach().cpu().numpy()
    valid_count = np.maximum(mask.sum(axis=1).astype(int), 1)
    last_idx = np.clip(valid_count - 1, 0, r_obs.shape[1] - 1)
    last = r_obs[np.arange(r_obs.shape[0]), last_idx]
    r_filled = np.where(mask > 0, r_obs, last[:, None])
    target = np.concatenate([
        r_filled[:, idx],
        np.maximum.accumulate(r_filled[:, idx], axis=1),
        csr[:, idx],
        np.diff(r_filled[:, idx], axis=1, prepend=r_filled[:, idx[:1]]),
    ], axis=1)

    pred_parts = [outputs['traj_mean'][:, idx]]
    for key in ['z', 'g', 'c']:
        if key in outputs:
            pred_parts.append(outputs[key][:, idx])
    pred_parts.append(np.diff(outputs['traj_mean'][:, idx], axis=1, prepend=outputs['traj_mean'][:, idx[:1]]))
    pred = np.concatenate(pred_parts, axis=1)
    return pred, target


def _finite_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    return np.nan_to_num(x, nan=0.0, posinf=1e6, neginf=-1e6)


def _standardize_np(x: np.ndarray) -> np.ndarray:
    x = _finite_np(x)
    z = (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-9)
    return np.clip(np.nan_to_num(z, nan=0.0, posinf=20.0, neginf=-20.0), -20.0, 20.0)


def _pairwise_np(x: np.ndarray) -> np.ndarray:
    z = _standardize_np(x)
    diff = z[:, None, :] - z[None, :, :]
    return np.sqrt(np.maximum((diff * diff).sum(axis=-1), 0.0))


def _knn_overlap(d_a: np.ndarray, d_b: np.ndarray, k: int = 12) -> float:
    n = d_a.shape[0]
    kk = min(k, n - 1)
    if kk <= 0:
        return np.nan
    a = np.argsort(d_a, axis=1)[:, 1:kk + 1]
    b = np.argsort(d_b, axis=1)[:, 1:kk + 1]
    return float(np.mean([len(set(a[i]).intersection(set(b[i]))) / kk for i in range(n)]))


def _h0_death_times(distance_matrix: np.ndarray) -> np.ndarray:
    # 0D persistence for a Vietoris-Rips filtration equals the MST edge lengths.
    d = _finite_np(distance_matrix)
    n = d.shape[0]
    if n <= 1:
        return np.array([], dtype=float)
    parent = np.arange(n)
    rank = np.zeros(n, dtype=int)

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a: int, b: int) -> bool:
        ra, rb = find(a), find(b)
        if ra == rb:
            return False
        if rank[ra] < rank[rb]:
            ra, rb = rb, ra
        parent[rb] = ra
        if rank[ra] == rank[rb]:
            rank[ra] += 1
        return True

    edges = [(float(d[i, j]), i, j) for i in range(n) for j in range(i + 1, n)]
    edges.sort(key=lambda x: x[0])
    deaths = []
    for weight, i, j in edges:
        if union(i, j):
            deaths.append(weight)
            if len(deaths) == n - 1:
                break
    return np.asarray(deaths, dtype=float)


def _h0_death_l2(d_a: np.ndarray, d_b: np.ndarray) -> float:
    a = _h0_death_times(d_a)
    b = _h0_death_times(d_b)
    m = max(len(a), len(b))
    if m == 0:
        return np.nan
    aa = np.pad(np.sort(a), (0, m - len(a)), constant_values=0.0)
    bb = np.pad(np.sort(b), (0, m - len(b)), constant_values=0.0)
    return float(np.linalg.norm(aa - bb) / np.sqrt(m))


def topology_diagnostics(outputs: Dict[str, np.ndarray], split: Dict[str, object], k: int = 12) -> Dict[str, float]:
    pred, target = _numpy_signature_from_split(split, outputs, n_points=TOPOLOGY_POINTS)
    pred_z = _standardize_np(pred)
    target_z = _standardize_np(target)
    d_pred = _pairwise_np(pred_z)
    d_target = _pairwise_np(target_z)
    iu = np.triu_indices_from(d_pred, k=1)
    if np.std(d_pred[iu]) < 1e-12 or np.std(d_target[iu]) < 1e-12:
        corr = np.nan
    else:
        corr = np.corrcoef(d_pred[iu], d_target[iu])[0, 1]
    out = {
        'topo_distance_corr': float(corr),
        'topo_knn_overlap': _knn_overlap(d_target, d_pred, k=k),
        'topo_h0_death_l2': _h0_death_l2(d_target, d_pred),
    }
    try:
        import warnings
        from sklearn.manifold import trustworthiness
        kk = min(k, len(pred_z) - 1)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            out['topo_trustworthiness'] = float(trustworthiness(target_z, pred_z, n_neighbors=kk)) if kk > 0 else np.nan
    except Exception:
        out['topo_trustworthiness'] = np.nan
    return out


def evaluate_model_with_topology(model, name: str, split_name: str = 'test') -> Tuple[Dict[str, float], pd.DataFrame]:
    split = benchmark[split_name]
    outputs = collect_outputs(model, split, cfg, device)
    metrics, sample = compute_metrics(name, outputs, split, cfg)
    metrics.update(topology_diagnostics(outputs, split, k=TOPOLOGY_K))
    return metrics, sample


## Baseline EVT-NeuralSSM: стандартные и topology-метрики

Сначала фиксируем baseline без переобучения. Это контроль: topology-regularized модель должна улучшать `topo_distance_corr`/`topo_knn_overlap` без заметной деградации `PPR RMSE`, `Brier`, `N_liq`.


In [ ]:
base_metrics, base_samples = evaluate_model_with_topology(base_evt, 'EVT-NeuralSSM', split_name='test')
focus_cols = [
    'model', 'N_liq_MAE', 'N_liq_logMAE', 'Brier', 'ECE', 'Traj_RMSE', 'Traj_CRPS',
    'CRR_RMSE', 'Physics_Violation_Rate', 'topo_distance_corr', 'topo_knn_overlap', 'topo_h0_death_l2', 'topo_trustworthiness'
]
pd.DataFrame([base_metrics])[focus_cols]


## Опциональное дообучение EVT+Topology

Блок ниже выключен по умолчанию. При `RUN_TRAINING=True` он:

1. создаёт `EVTTopoSSM` из весов обычного `evt_ssm`;
2. fine-tune-ит только EVT-ветку с `λL_topology`;
3. оценивает стандартные и topology-метрики;
4. сохраняет отдельную модель `models/evt_ssm_toporeg`.

DPI-Flow и DPI-EVT не загружаются и не изменяются.


In [ ]:
comparison_rows = [base_metrics]
toporeg_history = None

if RUN_TRAINING:
    toporeg = make_toporeg_model(load_evt_weights=True)
    toporeg, toporeg_history = train_model(
        toporeg,
        benchmark['train'],
        benchmark['val'],
        epochs=TOPOLOGY_EPOCHS,
        model_name='EVT-NeuralSSM+Topology',
        config=cfg,
        device=device,
        track_metrics=True,
        verbose=True,
        scheduler='cosine',
    )
    topo_metrics, topo_samples = evaluate_model_with_topology(toporeg, 'EVT-NeuralSSM+Topology', split_name='test')
    comparison_rows.append(topo_metrics)

    topo_hp = copy.deepcopy(hp)
    topo_hp['display_name'] = 'EVT-NeuralSSM+Topology'
    topo_hp['model_type'] = 'EVTTopoSSM (notebook-local wrapper)'
    topo_hp['topology_regularization'] = {
        'lambda': TOPOLOGY_LAMBDA,
        'k': TOPOLOGY_K,
        'points': TOPOLOGY_POINTS,
        'loss': 'weighted pairwise distance preservation between observed trajectories and predicted state-space trajectories',
    }
    save_trained_model(toporeg, REPO / 'models', TOPO_MODEL_NAME, topo_hp, toporeg_history)
else:
    print('RUN_TRAINING=False: показываем baseline и готовый код эксперимента без изменения models/.')

comparison = pd.DataFrame(comparison_rows)[focus_cols]
comparison


## Визуальная проверка: observed topology vs EVT state topology

Графики матриц расстояний позволяют увидеть, сохраняет ли state-space модель кластерную структуру траекторий. После включения `RUN_TRAINING=True` сюда можно добавить строку для `EVT-NeuralSSM+Topology` и сравнить локальные блоки.


In [ ]:
def plot_distance_matrices(model, title: str, split_name: str = 'test'):
    split = benchmark[split_name]
    outputs = collect_outputs(model, split, cfg, device)
    pred, target = _numpy_signature_from_split(split, outputs, n_points=TOPOLOGY_POINTS)
    d_target, d_pred = _pairwise_np(target), _pairwise_np(pred)
    order = np.argsort(split['meta']['liq_label'].to_numpy())

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
    for ax, mat, name in [
        (axes[0], d_target[np.ix_(order, order)], 'Observed PPR/CSR geometry'),
        (axes[1], d_pred[np.ix_(order, order)], f'{title} state geometry'),
        (axes[2], np.abs(d_pred - d_target)[np.ix_(order, order)], '|Δ distance|'),
    ]:
        im = ax.imshow(mat, cmap='viridis', aspect='auto')
        ax.set_title(name, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    return fig

fig = plot_distance_matrices(base_evt, 'EVT-NeuralSSM')
fig


## Optional PH diagnostics

Если в окружении стоят `ripser`/`kmapper`, можно дополнительно сравнить persistent summaries для observed и state signatures. Если зависимостей нет, ячейка мягко пропускает PH-часть; это не влияет на обучение, потому что `L_topology` полностью differentiable и не зависит от PH-библиотек.


In [ ]:
def optional_persistence_check(model, split_name: str = 'test', n_subsample: int = 120) -> pd.DataFrame:
    try:
        from liquefaction_ai.topology import compute_persistence, persistence_summary
    except Exception as exc:
        return pd.DataFrame([{'status': f'PH skipped: {type(exc).__name__}: {exc}'}])

    split = benchmark[split_name]
    outputs = collect_outputs(model, split, cfg, device)
    pred, target = _numpy_signature_from_split(split, outputs, n_points=TOPOLOGY_POINTS)
    rows = []
    try:
        for name, sig in [('observed', target), ('evt_state', pred)]:
            dgms = compute_persistence(_standardize_np(sig), maxdim=1, n_subsample=n_subsample, seed=42)
            row = {'space': name, 'status': 'ok'}
            row.update(persistence_summary(dgms))
            rows.append(row)
    except Exception as exc:
        return pd.DataFrame([{'status': f'PH skipped: {type(exc).__name__}: {exc}'}])
    return pd.DataFrame(rows)

optional_persistence_check(base_evt)


## Интерпретация результата

Топологическая регуляризация считается полезной, если после `RUN_TRAINING=True` выполняется хотя бы одно из условий без деградации основных метрик:

- растёт `topo_knn_overlap`: соседства наблюдаемых траекторий лучше сохраняются в state-space;
- растёт `topo_distance_corr`: глобальная геометрия траекторий ближе к observed geometry;
- падает `topo_h0_death_l2`: 0D persistence death-profile predicted state space ближе к observed trajectories;
- `Physics_Violation_Rate` остаётся нулевой;
- `Traj_RMSE`, `Brier`, `N_liq_logMAE` не ухудшаются сверх малой tolerance.

Если topology-метрики растут, а RMSE ухудшается, значит `λ` слишком велик: модель слишком жёстко сохраняет геометрию батча и теряет pointwise-подгонку. Тогда исследовать сетку `λ ∈ {0.005, 0.01, 0.03, 0.05}` и kNN `k ∈ {8, 12, 20}`.


In [ ]:
summary = comparison.copy()
num_cols = [c for c in summary.columns if c != 'model']
summary[num_cols] = summary[num_cols].apply(pd.to_numeric, errors='coerce')
summary.round(4)
